# 07 - Neo4j End-to-End (Reference)

This notebook demonstrates the full Neo4j workflow with orthograph: defining a model, populating a database, inspecting it, validating the schema, and validating query results.

**Requirements**: A running Neo4j instance (e.g. `bolt://localhost:7687`). This notebook is written as a **reference** -- all cells include explanatory markdown and expected output descriptions so it can be read without a live database. Code cells have no outputs because they require a live connection.

In [32]:
from typing import Optional

from orthograph.api.database import inspect, validate
from orthograph.backends.neo4j.result_adapter import (
    validate_result,
)
from orthograph.cypher.generator import CypherGenerator
from orthograph.cypher.parser import validate_cypher
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import NodeModel, RelationshipModel

## Define the data model

A simple filmography model: `Person` nodes connected to `Movie` nodes via `ACTED_IN` relationships.

In [33]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    role: str


graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)

print("Model:", graph_definition.name)
print("Nodes:", graph_definition.node_labels)
print("Rels: ", graph_definition.relationship_labels)

Model: Filmography
Nodes: {'Movie', 'Person'}
Rels:  {'ACTED_IN'}


Expected output:
```
Model: Filmography
Nodes: {'Person', 'Movie'}
Rels:  {'ACTED_IN'}
```

## Connect to Neo4j

Replace the URI and credentials below with your own Neo4j instance details.

In [34]:
from neo4j import GraphDatabase
from utils import load_env, print_apoc_status


# Load credentials from .env file (or .env_default if .env doesn't exist)
neo4j_uri = load_env("NEO4J_URI", "bolt://localhost:7687")
neo4j_user = load_env("NEO4J_USER", "neo4j")
neo4j_password = load_env("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# Check APOC availability
print_apoc_status(driver)

  APOC: ✓ Available (property types will be detected)


## Populate the database

Use `CypherGenerator` to produce Cypher statements from the model definition. We generate uniqueness constraints, merge nodes, and create relationships. The generated Cypher is printed so you can inspect it before execution.

In [35]:
gen = CypherGenerator(graph_definition)

# Step 1: Create uniqueness constraints
print("=== Constraints ===")
for stmt in gen.generate_constraints():
    print(stmt)
    driver.execute_query(stmt)
print()

# Step 2: Merge nodes
print("=== Nodes ===")
people = [
    {"__label__": "Person", "name": "Keanu Reeves", "born": 1964},
    {"__label__": "Person", "name": "Carrie-Anne Moss", "born": 1967},
    {"__label__": "Person", "name": "Lana Wachowski", "born": 1965},
]
movies = [
    {"__label__": "Movie", "title": "The Matrix", "year": 1999},
    {"__label__": "Movie", "title": "The Matrix Reloaded", "year": 2003},
]

for node_data in people + movies:
    query, params = gen.merge_node(node_data)
    print(f"  {query}  params={params}")
    driver.execute_query(query, **params)
print()

# Step 3: Create relationships
print("=== Relationships ===")
relationships = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix",
        "role": "Neo",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Carrie-Anne Moss",
        "__target_uid__": "The Matrix",
        "role": "Trinity",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix Reloaded",
        "role": "Neo",
    },
]

for rel_data in relationships:
    query, params = gen.create_relationship(rel_data)
    print(f"  {query}  params={params}")
    driver.execute_query(query, **params)

=== Constraints ===
CREATE CONSTRAINT constraint_person_name IF NOT EXISTS FOR (n:Person) REQUIRE n.name IS UNIQUE
CREATE CONSTRAINT constraint_movie_title IF NOT EXISTS FOR (n:Movie) REQUIRE n.title IS UNIQUE

=== Nodes ===
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n  params={'name': 'Keanu Reeves', 'born': 1964}
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n  params={'name': 'Carrie-Anne Moss', 'born': 1967}
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n  params={'name': 'Lana Wachowski', 'born': 1965}
  MERGE (n:Movie {title: $title}) SET n.year = $year RETURN n  params={'title': 'The Matrix', 'year': 1999}
  MERGE (n:Movie {title: $title}) SET n.year = $year RETURN n  params={'title': 'The Matrix Reloaded', 'year': 2003}

=== Relationships ===
  MATCH (a:Person {name: $src_uid}), (b:Movie {title: $tgt_uid}) CREATE (a)-[r:ACTED_IN {role: $role}]->(b) RETURN r  params={'src_uid': 'Keanu Reeves', 'tgt_uid': 'The Matrix', 'role': 'Neo'}
  MA

Expected output:
```
=== Constraints ===
CREATE CONSTRAINT constraint_person_name IF NOT EXISTS FOR (n:Person) REQUIRE n.name IS UNIQUE
CREATE CONSTRAINT constraint_movie_title IF NOT EXISTS FOR (n:Movie) REQUIRE n.title IS UNIQUE

=== Nodes ===
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n  params={'name': 'Keanu Reeves', 'born': 1964}
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n  params={'name': 'Carrie-Anne Moss', 'born': 1967}
  MERGE (n:Person {name: $name}) SET n.born = $born RETURN n  params={'name': 'Lana Wachowski', 'born': 1965}
  MERGE (n:Movie {title: $title}) SET n.year = $year RETURN n  params={'title': 'The Matrix', 'year': 1999}
  MERGE (n:Movie {title: $title}) SET n.year = $year RETURN n  params={'title': 'The Matrix Reloaded', 'year': 2003}

=== Relationships ===
  MATCH (a:Person {name: $src_uid}), (b:Movie {title: $tgt_uid}) CREATE (a)-[r:ACTED_IN {role: $role}]->(b) RETURN r  ...
  ...
```

## Inspect the database

The `inspect("neo4j", driver)` seam queries the database to produce a `GraphProfile`. It automatically detects whether APOC is available and chooses the appropriate query strategy. The profile contains node type profiles, relationship type profiles, property completeness stats, and constraint information.

In [36]:
profile = inspect("neo4j", driver)

print(profile.model_dump_json(indent=2))

{
  "source": "neo4j",
  "timestamp": "2026-06-16T23:50:57.541988",
  "node_type_profiles": {
    "Movie": {
      "label": "Movie",
      "count": 3,
      "property_profiles": {
        "title": {
          "name": "title",
          "present_count": 3,
          "total_count": 3,
          "observed_types": [
            "String"
          ],
          "observed_type_counts": {},
          "distinct_count": null,
          "missing_count": 0,
          "completeness": 1.0,
          "is_required": true
        },
        "year": {
          "name": "year",
          "present_count": 3,
          "total_count": 3,
          "observed_types": [
            "Long"
          ],
          "observed_type_counts": {},
          "distinct_count": null,
          "missing_count": 0,
          "completeness": 1.0,
          "is_required": true
        }
      }
    },
    "Person": {
      "label": "Person",
      "count": 4,
      "property_profiles": {
        "name": {
          "name": "n

Expected output (abbreviated): a JSON object with `source: "neo4j"`, `node_type_profiles` for Person and Movie, `rel_type_profiles` for ACTED_IN, `constraints` listing the uniqueness constraints, and property profiles with completeness and type information.

## Review the profile

Iterate the profile's node type profiles to see instance counts and property completeness. Key things to look for:

- **Property completeness**: are required properties present on all instances? A completeness below 100% for a required property indicates data quality issues.
- **Observed types**: do the observed database types match the expected Python types? A mismatch (e.g. `String` vs `int`) indicates a type conflict.
- **Cardinality stats**: does the observed min/max degree match the model's cardinality constraints?

In [37]:
for label, ntp in profile.node_type_profiles.items():
    print(f"=== {label} ({ntp.count} instances) ===")
    print(f"  {'Property':<15s} {'Completeness':>12s}  {'Types'}")
    print(f"  {'-' * 15} {'-' * 12}  {'-' * 20}")
    for prop_name, pp in ntp.property_profiles.items():
        print(f"  {prop_name:<15s} {pp.completeness:>11.0%}  {pp.observed_types}")
    print()

=== Movie (3 instances) ===
  Property        Completeness  Types
  --------------- ------------  --------------------
  title                  100%  ['String']
  year                   100%  ['Long']

=== Person (4 instances) ===
  Property        Completeness  Types
  --------------- ------------  --------------------
  name                   100%  ['String']
  born                   100%  ['Long']



Expected output:
```
=== Person (3 instances) ===
  Property         Completeness  Types
  --------------- ------------  --------------------
  born                    100%  ['Long']
  name                    100%  ['String']

=== Movie (2 instances) ===
  Property         Completeness  Types
  --------------- ------------  --------------------
  title                   100%  ['String']
  year                    100%  ['Long']
```

## Validate against the model

Passing a `model` to `inspect("neo4j", driver, model)` produces a profile and runs `compare` against the model in a single call. It returns a `ValidationResult` with errors, warnings, and info-level issues.

In [38]:
result = validate("neo4j", driver, graph_definition)

print("is_valid:", result.is_valid)
print(f"Errors:   {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")

for issue in result.issues:
    print(f"  [{issue.severity.value}] [{issue.code}] {issue.message}")

is_valid: True
Errors:   0
Warnings: 0


Expected output (when the database matches the model):
```
is_valid: True
Errors:   0
Warnings: 0
```

If the database contained nodes or relationships not defined in the model, or if required properties were incomplete, you would see issues listed here.

## Validate a Cypher query

The `validate_cypher` function parses a Cypher query and checks that the labels and properties referenced in the query are consistent with the model. This catches errors like querying for properties that do not exist in the model.

In [39]:
# This query references 'salary', which is not a property on Person in our model
result = validate_cypher("MATCH (n:Person) RETURN n.salary", graph_definition)

print("Errors:")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

Errors:
  [QUERY_UNKNOWN_PROPERTY] Query accesses property 'salary' on Person which is not in the model


Expected output:
```
Errors:
  [UNKNOWN_PROPERTY] Property 'salary' not defined on node type 'Person'
```

## Validate query results

After executing a Cypher query, you can validate the returned records against the model. The `validate_result` function extracts nodes and relationships from the Neo4j driver result records and validates them using `GraphValidator`.

In [40]:
records, _, _ = driver.execute_query(
    "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) RETURN p, r, m"
)

result = validate_result(records, graph_definition)

print("is_valid:", result.is_valid)
print(f"Errors:   {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")

is_valid: True
Errors:   0
Warnings: 0


Expected output:
```
is_valid: True
Errors:   0
Warnings: 0
```

If the query returned nodes with unknown labels, missing required properties, or relationships with wrong endpoint types, those issues would be reported.

## Cleanup

Close the driver connection when done.

In [41]:
driver.close()